# CSE 151B Competition — Final Submission
Qwen3-4B-Thinking + Round 1 QLoRA adapter (alanj21/qwen3-4b-sft-round1)

Run cells top to bottom. Call `run_inference()` at the bottom to produce the submission file.
After a crash, re-run from the top — completed questions are skipped automatically.

## 1. Install Dependencies

In [ ]:
!pip install -q peft bitsandbytes>=0.46.1 sympy antlr4-python3-runtime==4.11.1

## 2. Imports

In [ ]:
import csv
import json
import re
import sys
from itertools import islice
from pathlib import Path
from typing import Optional

import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

sys.path.insert(0, "/kaggle/input/datasets/alanj21/dataset1")
from judger import Judger

## 3. Configuration

Edit only this cell.

In [ ]:
BASE_MODEL_ID   = "Qwen/Qwen3-4B-Thinking-2507"
ADAPTER_ID      = "a3jiang/qwen3-4b-sft-round1"   # HuggingFace Hub — Round 1 QLoRA adapter
DATA_PATH       = "/kaggle/input/datasets/alanj21/dataset1/private2.jsonl"
EXISTING_CHECKPOINT_PATH = "/kaggle/input/datasets/alanj21/dataset1/checkpoint_private.jsonl"
CHECKPOINT_PATH = "/kaggle/working/checkpoint_private.jsonl"
OUTPUT_PATH     = "/kaggle/working/submission.csv"   # final submission CSV

BATCH_SIZE      = 4       # lower to 2 if OOM
MAX_NEW_TOKENS  = 4000
TEMPERATURE     = 0.6
TOP_P           = 0.95
TOP_K           = 20
SAVE_EVAL       = False   # False for private set — no gold answers available
PRINT_THINKING  = False
THINK_PREVIEW_CHARS = 800

## 4. Prompts

In [ ]:
FEW_SHOT_MATH = (
    "\nEXAMPLE (free-form):\n"
    "Q: Find the sum of the first 10 positive even integers.\n"
    "Using the formula for sum of first n even integers: n(n+1) = 10*11 = 110.\n"
    "\\boxed{110}\n"
)

FEW_SHOT_MCQ = (
    "\nEXAMPLE (MCQ):\n"
    "Q: What is 25/40 in reduced form?\n"
    "Options:\n"
    "A. 1/2\nB. 3/4\nC. 5/8\nD. 2/3\n"
    "GCD(25,40)=5. 25/5=5, 40/5=8. Answer is 5/8.\n"
    "\\boxed{C}\n"
)

DOUBLE_CHECK_RULE = """
ANSWER VERIFICATION PROTOCOL (follow exactly):
Step 1 — Solve the problem. Call this Answer_1.
Step 2 — You have ONE chance to verify. Recompute independently. Call this Answer_2.
  - If Answer_2 == Answer_1: this is your FINAL ANSWER. Write </think> and output \\boxed{Answer_1}. STOP.
  - If Answer_2 != Answer_1: proceed to Step 3.
Step 3 — One final recompute. Call this Answer_3.
  - If Answer_3 == Answer_1: write </think> and output \\boxed{Answer_1}. STOP.
  - If Answer_3 == Answer_2: write </think> and output \\boxed{Answer_2}. STOP.
  - If Answer_3 != Answer_1 and Answer_3 != Answer_2: write </think> and output \\boxed{Answer_1}. STOP.
HARD LIMIT: After 3 computations you MUST write </think> and output a \\boxed{} answer.
You are NOT permitted to compute a 4th time under any circumstances.
"""

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}.\n\n"
    "RULES:\n"
    "- DO NOT USE FILLER WORDS! For example, 'okay', 'hmm', 'wait', 'let me check', or 'let me verify'.\n"
    "- When a formula below applies, USE IT DIRECTLY WITHOUT RE-DERIVING IT.\n"
    "- Once you reach an answer via a valid method, state it. Do not re-verify with alternative methods.\n\n"
    "MATH REFERENCE:\n"
    "- Sum of first n integers: n(n+1)/2\n"
    "- Sum of first n even integers: n(n+1)\n"
    "- Sum of first n odd integers: n^2\n"
    "- Arithmetic sequence sum: (n/2)(first + last)\n"
    "- Geometric series sum (finite): a(1 - r^n)/(1 - r)\n"
    "- Integral of 1/(x^2 + a^2) dx = (1/a)arctan(x/a) + C\n"
    "- Integral over all reals of 1/(x^2 + a^2) dx = pi/a\n"
    "- Newton's Law of Cooling: T(t) = T_s + (T_0 - T_s)e^(-kt)\n"
    "- GCD: divide both numbers by their largest common prime factor\n"
    "- To reduce a fraction: divide numerator and denominator by GCD\n"
) + DOUBLE_CHECK_RULE + FEW_SHOT_MATH

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}.\n\n"
    "RULES:\n"
    "- When a formula below applies, use it directly without re-deriving it.\n"
    "- Once you identify the answer, state it immediately. Do not re-check unless you made an arithmetic error.\n\n"
    "MATH REFERENCE:\n"
    "- Sum of first n integers: n(n+1)/2\n"
    "- Sum of first n even integers: n(n+1)\n"
    "- Sum of first n odd integers: n^2\n"
    "- Arithmetic sequence sum: (n/2)(first + last)\n"
    "- Geometric series sum (finite): a(1 - r^n)/(1 - r)\n"
    "- Integral of 1/(x^2 + a^2) dx = (1/a)arctan(x/a) + C\n"
    "- Integral over all reals of 1/(x^2 + a^2) dx = pi/a\n"
    "- Newton's Law of Cooling: T(t) = T_s + (T_0 - T_s)e^(-kt)\n"
    "- GCD: divide both numbers by their largest common prime factor\n"
) + DOUBLE_CHECK_RULE + FEW_SHOT_MCQ

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

## 5. Load Model + Adapter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # critical for batched decoder-only generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(model, ADAPTER_ID)
model.eval()

print("Base model + Round 1 adapter loaded.")
print(f"Device map: {model.base_model.model.hf_device_map}")

## 6. Scoring Helpers

In [ ]:
judger = Judger(strict_extract=False)

def split_thinking(response: str) -> tuple[str, str]:
    tag = "</think>"
    idx = response.rfind(tag)
    if idx == -1:
        return "", response.strip()
    return response[:idx].replace("<think>", "").strip(), response[idx + len(tag):].strip()

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_response(response: str, gold, is_mcq: bool) -> bool:
    if is_mcq:
        return extract_letter(response) == str(gold).strip().upper()
    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
    except Exception:
        return False

def print_thinking(item: dict, response: str, correct: bool) -> None:
    thinking, final = split_thinking(response)
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"ID: {item['id']}  |  Correct: {correct}")
    print(f"QUESTION: {item['question'][:300]}")
    if thinking:
        truncated = len(thinking) > THINK_PREVIEW_CHARS
        print(f"\n[THINKING — {len(thinking)} chars{'  (truncated below)' if truncated else ''}]")
        print(thinking[:THINK_PREVIEW_CHARS])
    else:
        print("\n[NO THINKING BLOCK]")
    print(f"\n[FINAL ANSWER]\n{final}")
    if SAVE_EVAL:
        print(f"[GOLD] {item['answer']}")

## 7. run_inference()

In [ ]:
def run_inference(
    data_path=DATA_PATH,
    existing_checkpoint_path = EXISTING_CHECKPOINT_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    output_path=OUTPUT_PATH,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
):
    # ── Load dataset ──────────────────────────────────────────────────────
    data = [json.loads(line) for line in open(data_path)]
    n_mcq  = sum(bool(d.get("options")) for d in data)
    n_free = len(data) - n_mcq
    print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

    # ── Resume from checkpoint ────────────────────────────────────────────
    completed = {}
    if Path(existing_checkpoint_path).exists():
        with open(existing_checkpoint_path) as f:
            for line in f:
                rec = json.loads(line)
                completed[rec["id"]] = rec

    remaining = [d for d in data if d["id"] not in completed]
    print(f"Already completed: {len(completed)} / {len(data)}")
    print(f"Remaining:         {len(remaining)}")

    def batched(iterable, n):
        it = iter(iterable)
        while chunk := list(islice(it, n)):
            yield chunk

    # ── Generate ──────────────────────────────────────────────────────────
    with open(checkpoint_path, "a") as ckpt_file:
        for batch in tqdm(
            batched(remaining, batch_size),
            total=-(-len(remaining) // batch_size),
            desc="Generating",
        ):
            prompts = []
            for item in batch:
                system, user = build_prompt(item["question"], item.get("options"))
                prompt_text = tokenizer.apply_chat_template(
                    [{"role": "system", "content": system},
                     {"role": "user",   "content": user}],
                    tokenize=False,
                    add_generation_prompt=True,
                )
                prompts.append(prompt_text)

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024,
            ).to(model.device)

            prompt_len = inputs["input_ids"].shape[1]

            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    top_k=TOP_K,
                    repetition_penalty=1.0,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            responses = [
                tokenizer.decode(out[prompt_len:], skip_special_tokens=True).strip()
                for out in output_ids
            ]

            for item, response in zip(batch, responses):
                is_mcq = bool(item.get("options"))
                if SAVE_EVAL:
                    correct = score_response(response, item["answer"], is_mcq)
                    if PRINT_THINKING:
                        print_thinking(item, response, correct)
                    record = {"id": item["id"], "is_mcq": is_mcq,
                              "gold": item["answer"], "response": response, "correct": correct}
                else:
                    if PRINT_THINKING:
                        print_thinking(item, response, False)
                    record = {"id": item["id"], "is_mcq": is_mcq, "response": response}

                ckpt_file.write(json.dumps(record) + "\n")

            ckpt_file.flush()

    print("Generation complete.")

    # ── Collect results ───────────────────────────────────────────────────
    results = []
    with open(checkpoint_path) as f:
        for line in f:
            results.append(json.loads(line))

    # ── Save CSV ──────────────────────────────────────────────────────────
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "response"])
        for r in results:
            writer.writerow([r["id"], r["response"]])

    print(f"Saved {len(results)} records → {output_path}")

    # ── Summary (only if SAVE_EVAL) ───────────────────────────────────────
    if SAVE_EVAL:
        mcq_res  = [r for r in results if r["is_mcq"]]
        free_res = [r for r in results if not r["is_mcq"]]
        def acc(s): return sum(r["correct"] for r in s) / len(s) * 100 if s else 0.0
        print("=" * 50)
        print(f"  MCQ       : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
        print(f"  Free-form : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
        print(f"  Overall   : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
        print("=" * 50)

    return results

## 8. Run

In [ ]:
results = run_inference()